# TorchAX + HuggingFace Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_quickstart.ipynb)

Run a HuggingFace model on JAX in under 10 cells.

**Full tutorial:** [torchax_huggingface_tutorial.ipynb](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb) | [Blog post](https://dev.to/ahmed_elnaggar/run-any-huggingface-model-on-tpus-a-beginners-guide-to-torchax)

In [ ]:
# Install dependencies (auto-detect TPU/GPU)
import subprocess
try:
    subprocess.run(["ls", "/dev/accel0"], capture_output=True, check=True)
    accel = "tpu"
except Exception:
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True)
        accel = "gpu"
    except Exception:
        accel = "cpu"

!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
if accel == "tpu":
    !pip install -q -U jax[tpu]
elif accel == "gpu":
    !pip install -q -U jax[cuda12]
else:
    !pip install -q -U jax
!pip install -q -U torchax transformers flax
print(f"Accelerator: {accel}")

In [ ]:
import torch, torchax, jax, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="cpu")

# Enable torchax AFTER model loading to avoid intercepting unsupported init ops
torchax.enable_globally()
model.to("jax")

print(f"Loaded {MODEL} on {jax.devices()}")

In [ ]:
# Eager forward pass
prompt = "The secret to baking a good cake is"
ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to("jax")

start = time.perf_counter()
with torch.no_grad():
    out = model(ids, use_cache=False)
print(f"Eager: {time.perf_counter()-start:.3f}s")
print(f"Next token: '{tokenizer.decode([torch.argmax(out.logits[0,-1]).item()])}'")

In [ ]:
# JIT-compiled forward pass
from jax.tree_util import register_pytree_node
from transformers import modeling_outputs, cache_utils

register_pytree_node(modeling_outputs.CausalLMOutputWithPast, lambda v: (v.to_tuple(), None), lambda a, c: modeling_outputs.CausalLMOutputWithPast(*c))
register_pytree_node(cache_utils.DynamicCache, lambda c: ((c.key_cache, c.value_cache), None), lambda a, c: setattr(cache_utils.DynamicCache(), 'key_cache', c[0]) or setattr(cache_utils.DynamicCache(), 'value_cache', c[1]) or cache_utils.DynamicCache())

weights, jax_func = torchax.extract_jax(model)
jitted = jax.jit(lambda w, x: jax_func(w, (x,), {"use_cache": False}))

# Convert input to native JAX array for jax.jit
jax_ids = jax.device_put(ids.cpu().numpy())

# Warm up
res = jitted(weights, jax_ids); jax.block_until_ready(res)

# Benchmark
for i in range(3):
    s = time.perf_counter()
    res = jitted(weights, jax_ids); jax.block_until_ready(res)
    print(f"JIT run {i}: {time.perf_counter()-s:.4f}s")

In [ ]:
# Text generation with StaticCache
from transformers.cache_utils import StaticCache

register_pytree_node(
    StaticCache,
    lambda c: ((c.key_cache, c.value_cache),
               (c.config, c.max_batch_size, c.max_cache_len,
                getattr(c, "device", None), getattr(c, "dtype", None))),
    lambda a, c: (lambda sc: (setattr(sc, 'key_cache', c[0]), setattr(sc, 'value_cache', c[1]), sc)[-1])(
        StaticCache(a[0], a[1], a[2], **({} if a[3] is None else {"device": a[3]}), **({} if a[4] is None else {"dtype": a[4]}))
    ),
)

def generate(prompt, max_tokens=50):
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to("jax")
    bs, sl = ids.shape
    kv = StaticCache(config=model.config, max_batch_size=1, max_cache_len=sl+max_tokens, device="jax", dtype=model.dtype)
    pos = torch.arange(sl, device="jax")

    with torch.no_grad():
        logits, kv = model(ids, cache_position=pos, past_key_values=kv, return_dict=False, use_cache=True)

    tok = torch.argmax(logits[:,-1], dim=-1)[:,None]
    out = [tok[:,0].item()]
    pos = torch.tensor([sl], device="jax")

    for _ in range(max_tokens-1):
        with torch.no_grad():
            logits, kv = model(tok, cache_position=pos, past_key_values=kv, return_dict=False, use_cache=True)
        tok = torch.argmax(logits[:,-1], dim=-1)[:,None]
        tid = tok[:,0].item()
        if tid == tokenizer.eos_token_id: break
        out.append(tid)
        pos += 1

    return tokenizer.decode(out, skip_special_tokens=True)

print(generate("The secret to baking a good cake is"))

In [ ]:
# Benchmark generation
for n in [10, 25, 50]:
    s = time.perf_counter()
    result = generate("Explain quantum computing in one paragraph.", max_tokens=n)
    elapsed = time.perf_counter() - s
    print(f"{n} tokens: {elapsed:.2f}s ({n/elapsed:.1f} tok/s)")

In [ ]:
# Mini chatbot
def chat(msg, max_tokens=100):
    prompt = f"<start_of_turn>user\n{msg}<end_of_turn>\n<start_of_turn>model\n"
    return generate(prompt, max_tokens)

for q in ["What is JAX?", "Write a haiku about GPUs."]:
    print(f"User: {q}")
    print(f"Gemma: {chat(q)}\n")

## Want to learn more?

- [Full tutorial notebook](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb) — detailed explanations of every concept
- [Blog post](https://dev.to/ahmed_elnaggar/run-any-huggingface-model-on-tpus-a-beginners-guide-to-torchax) — the complete written tutorial
- [torchax GitHub](https://github.com/google/torchax) — library source
- [Original tutorial series](https://huggingface.co/blog/qihqi/huggingface-jax-01) — by Han Qi, author of torchax